In [1]:
import pandas as pd
import re
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score,classification_report

In [2]:
# Load the dataset
data=pd.read_csv('IMDB Dataset.csv')

# check for the null value
print(f"Any null values: \n{data.isnull().sum()} ")

# check dataset shape
print(f"\nDataset shape: {data.shape} ")

# top 5 rows
print(f"\nTop 5 rows in dataset:\n{data.head()} ")

# find number of samples for each label
print(data['sentiment'].value_counts())

# define text processing function
def clean_text(text):
    text=text.lower() # convert into lowercase
    text=re.sub(r"<.*?>", "",text) # remove html tag
    text=re.sub(r"http\S+|www\S+","",text) # remove link
    text=re.sub(r"[^a-z\s]","",text) # keep all lowercase letter and tab,whitespace,space and remove others
    text=text.strip() # remove white space
    return text

# apply text processing
data['review']=data['review'].apply(clean_text)

# split into feature and label
x=data['review']
y=data['sentiment'].map({"positive":1,"negative":0})

# split into test and train set
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

Any null values: 
review       0
sentiment    0
dtype: int64 

Dataset shape: (50000, 2) 

Top 5 rows in dataset:
                                              review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive 
sentiment
positive    25000
negative    25000
Name: count, dtype: int64


# What is TfidfVectorizer?
- Converts text into numerical features that can be given as input to a machine learning model.
- TF (Term Frequency): Measures how frequently a word appears in one review.
- IDF (Inverse Document Frequency): Measures how rare a word is across all training reviews.
- TF-IDF = TF × IDF: Calculates a score for each vocabulary word in each review based on its frequency and rarity.
- If a word does not appear in a review, its TF-IDF score for that review is 0.
- Words that appear frequently in a particular review but are rare across other reviews generally receive higher scores.
- Words appearing in almost every review generally receive lower IDF values.
- The model uses these numerical features along with the sentiment labels (positive = 1, negative = 0) to learn patterns for each class.
- max_features=5000: Selects a maximum of 5,000 vocabulary words as features, based on term frequency.
- stop_words="english": Removes common English words like "is", "was", "the", and "then".

In [4]:
# Convert text to number using TF IDF
vectorizer=TfidfVectorizer(max_features=5000,stop_words=None)

# fit and tranform the train and test 
x_train_vectorization=vectorizer.fit_transform(x_train)
x_test_vectorization=vectorizer.transform(x_test)

# define model
model=LogisticRegression(max_iter=200)


# train model
model.fit(x_train_vectorization,y_train)

# Evaluate model 
y_predict=model.predict(x_test_vectorization)
print(f"Accuracy : {accuracy_score(y_test,y_predict)} ")
print(classification_report(
    y_test,
    y_predict,
    target_names=["negative", "positive"]
))

Accuracy : 0.8896 
              precision    recall  f1-score   support

    negative       0.89      0.88      0.89      5000
    positive       0.89      0.90      0.89      5000

    accuracy                           0.89     10000
   macro avg       0.89      0.89      0.89     10000
weighted avg       0.89      0.89      0.89     10000

